# Member 1 — Handling Missing Data

**Technique:** Detect and handle missing or unreadable images before modelling.

## Why this dataset needs it
The basil dataset is organised as image folders, not a tidy CSV. “Missing data” here means:
- expected files listed for a region folder are absent
- files exist but **fail to decode** (corrupt / unsupported)
- an entire labelled folder is empty

Without this step, training would silently drop samples or crash mid-pipeline.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


## 1. Inventory expected vs found images


In [ ]:
inv = inventory_table(RAW)
display(inv)

missing_total = int(inv["missing_count"].sum())
print(f"Total missing vs expected file counts: {missing_total}")
print(f"Folders incomplete: {(~inv['complete']).sum()} / {len(inv)}")


## 2. Discover files and audit readability (treat failed reads as missing)


In [ ]:
discovered = discover_images(RAW)
print("Discovered image files:", len(discovered))

valid, rejected = audit_images(RAW, discovered)
print("Readable images:", len(valid))
print("Rejected / unreadable:", len(rejected))
if len(rejected):
    display(rejected[["path", "reason"]].head(20))

# Persist cleaned inventory for the group pipeline
valid.to_csv(OUT / "m1_valid_image_index.csv", index=False)
rejected.to_csv(OUT / "m1_rejected_images.csv", index=False)
inv.to_csv(OUT / "m1_folder_inventory.csv", index=False)

log = {
    "discovered": int(len(discovered)),
    "valid": int(len(valid)),
    "rejected": int(len(rejected)),
    "missing_vs_expected": missing_total,
}
(LOGS / "m1_missing_data.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
print("Saved outputs under results/outputs and results/logs")


## 3. EDA visualization — class balance & completeness


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class counts among readable images
class_counts = valid["label"].value_counts().reindex(CLASSES).fillna(0)
axes[0].bar(class_counts.index, class_counts.values, color=["#2ca02c", "#d62728"])
axes[0].set_title("Readable images per class")
axes[0].set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(int(v)), ha="center")

# Completeness by folder
axes[1].barh(inv["folder"], inv["found"], color="#1f77b4", label="Found")
axes[1].barh(inv["folder"], inv["missing_count"], left=inv["found"], color="#ff7f0e", label="Missing vs expected")
axes[1].set_title("Folder completeness (found + missing)")
axes[1].legend(loc="lower right")
axes[1].set_xlabel("Images")

fig.tight_layout()
fig.savefig(VIZ / "m1_class_and_completeness.png", dpi=150, bbox_inches="tight")
plt.show()
print("Interpretation: bars show whether both classes are present and whether any source folder is under-complete before later preprocessing.")


## Viva talking points
1. Define missing data for **images** (absent files + failed loads).
2. Show the inventory table and rejected list.
3. Interpret the class-balance / completeness chart.
